# Kehribar Video - Colab GPU Backend

Bu notebook, Kehribar Video Android uygulaması için gerekli olan
**Wan Image-to-Video** backend'ini Google Colab'ın ücretsiz/ücretli GPU'sunda
çalıştırır ve dışarıya açık bir URL üretir.

**Önce:** Üstte *Runtime > Change runtime type > T4 GPU* (veya daha güçlü bir GPU) seçin.


In [ ]:
# 1) GPU kontrolü
!nvidia-smi


In [ ]:
# 2) Proje dosyalarını çek
# Seçenek A: Bu repoyu GitHub'a yüklediyseniz clone edin:
# !git clone https://github.com/efecanca/kehribar-video.git
# %cd kehribar-video/backend

# Seçenek B: ZIP'i Colab'a manuel yükleyip (sol panel > Files) açın:
from google.colab import files
import zipfile, os

if not os.path.exists('backend'):
    print('Lütfen KehribarVideo.zip dosyasını sol paneldeki Files bölümüne sürükleyin,')
    print('sonra bu hücreyi tekrar çalıştırın.')
else:
    print('backend/ klasörü bulundu, devam edebilirsiniz.')


In [ ]:
# ZIP zaten yüklendiyse açmak için:
import zipfile, os
if os.path.exists('KehribarVideo.zip') and not os.path.exists('backend'):
    with zipfile.ZipFile('KehribarVideo.zip', 'r') as z:
        z.extractall('.')
    print('ZIP açıldı.')

%cd backend
!ls


In [ ]:
# 3) Bağımlılıkları kur (Colab'ın PyTorch/CUDA kurulumu genelde hazırdır,
# bu yüzden torch satırlarını burada tekrar kurmuyoruz; gerekirse requirements.txt
# içindeki torch sürümünü Colab'ın CUDA sürümüyle eşleştirin.)
!pip install -q fastapi uvicorn python-multipart diffusers transformers accelerate \
    safetensors sentencepiece ftfy imageio imageio-ffmpeg pyngrok nest_asyncio


In [ ]:
# 4) (İsteğe bağlı ama önerilir) Model ağırlıklarını önceden indir
# Böylece ilk /generate isteği modelin GPU'ya yüklenmesini beklemez.
from huggingface_hub import snapshot_download

MODEL_ID = "Wan-AI/Wan2.1-I2V-14B-480P"
snapshot_download(repo_id=MODEL_ID)
print("Model indirildi:", MODEL_ID)


In [ ]:
# 5) ngrok auth token
# https://dashboard.ngrok.com/get-started/your-authtoken adresinden alıp buraya yapıştırın.
NGROK_AUTH_TOKEN = "BURAYA_NGROK_TOKEN_YAPISTIRIN"  # @param {type:"string"}

from pyngrok import ngrok
if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "BURAYA_NGROK_TOKEN_YAPISTIRIN":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("UYARI: ngrok token girmediniz, ücretsiz/geçici tünel çalışmayabilir.")


### GPU belleği ve hız hakkında önemli not

`Wan2.1-I2V-14B`, ücretsiz Colab'ın T4 GPU'suna (16GB) çıplak haliyle sığmaz.
Bu yüzden aşağıdaki hücre `WAN_OFFLOAD_MODE=sequential` ayarlar; bu, OOM
(bellek yetersizliği) hatasını büyük ölçüde önler ama **tek bir video onlarca
dakika sürebilir**. Daha hızlı sonuç isterseniz Colab Pro'da L4/A100 GPU seçip
`WAN_OFFLOAD_MODE`'u `model` yapabilirsiniz (bkz. `backend/README.md`).

In [ ]:
import os

# T4 (16GB) için güvenli varsayılan: sequential offload + düşürülmüş kare sayısı.
# Daha güçlü bir GPU (L4/A100, Colab Pro) seçtiyseniz "model" olarak değiştirip
# hızlanabilirsiniz.
os.environ["WAN_OFFLOAD_MODE"] = "sequential"
os.environ["WAN_NUM_FRAMES"] = "33"


In [ ]:
# 6) FastAPI sunucusunu arka planda başlat ve ngrok ile dışarı aç
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

from main import app  # backend/main.py

public_url = ngrok.connect(8000)
print("=" * 60)
print("Android uygulamasi Ayarlar ekranina bu adresi yapistirin:")
print(public_url)
print("=" * 60)

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()


In [ ]:
# 7) Hızlı sağlık kontrolü
import time, requests
time.sleep(3)
r = requests.get("http://127.0.0.1:8000/health")
print(r.status_code, r.text)


## Notlar

- Colab oturumu kapanırsa (uzun süre etkileşimsiz kalırsa) bu backend durur ve
  ngrok URL'si geçersiz olur. Bu durumda notebook'u yeniden çalıştırıp yeni
  URL'yi Android uygulamasının **Ayarlar** ekranına güncellemeniz yeterlidir.
- Daha kararlı/kalıcı bir kurulum isterseniz, aynı `backend/` klasörünü kendi
  kalıcı bir CUDA GPU sunucunuzda (örn. bir GPU'lu VPS) `uvicorn main:app`
  ile çalıştırabilirsiniz; ngrok'a gerek kalmaz.
